In [1]:
from pathlib import Path
import duckdb
import requests
from tqdm import tqdm
import pandas as pd

In [2]:
DATA_DIR = Path("data")
CATEGORY = "Health_and_Personal_Care"
BASE_URL = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw"
REVIEWS_URL = f"{BASE_URL}/review_categories/{CATEGORY}.jsonl.gz"
META_URL    = f"{BASE_URL}/meta_categories/meta_{CATEGORY}.jsonl.gz"
REVIEWS_FILE = DATA_DIR / f"{CATEGORY}.jsonl.gz"
META_FILE    = DATA_DIR / f"meta_{CATEGORY}.jsonl.gz"
OUTPUT_FILE  = DATA_DIR / f"{CATEGORY}_merged.parquet"

In [3]:
c2 = duckdb.connect()

In [4]:
c2.execute(f"SELECT * FROM read_json_auto('{REVIEWS_URL}') LIMIT 5").df()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,4.0,12 mg is 12 on the periodic table people! Mg f...,This review is more to clarify someone else’s ...,[],B07TDSJZMR,B07TDSJZMR,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1580950175902,3,True
1,5.0,Save the lanet using less plastic.,Love these easy multitasking bleach tablets. B...,[],B08637FWWF,B08637FWWF,AEVWAM3YWN5URJVJIZZ6XPD2MKIA,1604354586880,3,True
2,5.0,Fantastic,I have been suffering a couple months with hee...,[],B07KJVGNN5,B07KJVGNN5,AHSPLDNW5OOUK2PLH7GXLACFBZNQ,1563966838905,0,True
3,4.0,It holds the water and makes bubbles. That's ...,"It's cheap and it does what I wanted. The ""ma...",[],B007HY7GC2,B092RP73CX,AEZGPLOYTSAPR3DHZKKXEFPAXUAA,1662258542725,7,True
4,1.0,Not for me,Didn't do a thing for me. Not saying they don'...,[],B08KYJLF5T,B08KYJLF5T,AEQAYV7RXZEBXMQIQPL6KCT2CFWQ,1642722787262,0,True


In [5]:
c2.execute(f"SELECT * FROM read_json_auto('{META_URL}') LIMIT 5").df()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,Health & Personal Care,Silicone Bath Body Brush Exfoliator Shower Bac...,3.9,7,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Rzoeox,[],"{'Package Dimensions': '""15 x 3.3 x 1.5 inches...",B07V346GZH,None
1,Health & Personal Care,"iPhone 7 Plus 8 Plus Screen Protector, ZHXIN T...",3.8,2,[Tough and Robust: Like all 78X screen protect...,[Features: 2.5D Arc Edge Treatment: The edge i...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],ZHXIN,[],"{'Brand': '""ZHXIN""', 'Compatible Devices': '""C...",B075W927RH,None
2,Health & Personal Care,Zig Zag Rolling Machine 70mm Size With FREE BO...,3.9,7,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],NaN,[],"{'Package Dimensions': '""4.1 x 1.8 x 0.3 inche...",B01FB26VKY,None
3,Health & Personal Care,Sting-Kill Disposable Wipes 8 Each ( Pack of 5),4.1,6,[],"[effective on stings and bites from bees, wasp...",21.37,[{'thumb': 'https://m.media-amazon.com/images/...,[],Sting-kill,[],"{'Brand': '""Sting-kill""', 'Item Form': '""Wipe""...",B01IAI29RU,None
4,Health & Personal Care,Heated Eyelash Curler Mini Portable Electric E...,3.3,8,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],BiBOSS,[],"{'Package Dimensions': '""6.1 x 3.1 x 1.9 inche...",B08CMN38RC,None


In [6]:
c2.execute(f"""
      COPY (SELECT * FROM read_json_auto('{REVIEWS_URL}')  LIMIT 20000)
      TO '../data/raw/reviews_raw.parquet'
      (FORMAT PARQUET, COMPRESSION ZSTD)
  """)

In [7]:
c2.execute(f"""
      COPY (SELECT * FROM read_json_auto('{META_URL}') LIMIT 20000)
      TO '../data/raw/meta_raw.parquet'
      (FORMAT PARQUET, COMPRESSION ZSTD)
  """)

In [8]:
c2.execute("""
    COPY (
        SELECT r.*, m.title AS product_title, m.price,
                    m.average_rating, m.main_category, m.store
        FROM read_parquet('../data/raw/reviews_raw.parquet') r
        LEFT JOIN read_parquet('../data/raw/meta_raw.parquet') m USING (parent_asin)
    )
    TO '../data/raw/merged.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

In [9]:
c2.execute(f"SELECT * FROM read_parquet('../data/raw/merged.parquet')").df()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,product_title,price,average_rating,main_category,store
0,4.0,It holds the water and makes bubbles. That's ...,"It's cheap and it does what I wanted. The ""ma...",[],B007HY7GC2,B092RP73CX,AEZGPLOYTSAPR3DHZKKXEFPAXUAA,1662258542725,7,True,"Homedics Bubble Bliss Deluxe-Foot Spa, Heat Ma...",NaN,4.4,Health & Personal Care,Homedics
1,1.0,Not for me,Didn't do a thing for me. Not saying they don'...,[],B08KYJLF5T,B08KYJLF5T,AEQAYV7RXZEBXMQIQPL6KCT2CFWQ,1642722787262,0,True,Brain Supplement 1053mg - Premium Nootropic Br...,NaN,4.1,Health & Personal Care,Nature's Nutrition
2,4.0,Makes a nice compact noise machine to take wit...,This is a nice basic sound machine. I have use...,[],B08THJD1MH,B08THJD1MH,AFSKPY37N3C43SOI5IEXEK5JSIYA,1617907534645,0,False,White Air Purifier and Dehumidifier Q10 True H...,NaN,2.8,Health & Personal Care,Afloia
3,5.0,Great hair dryer gets the job done quickly!,This Jinri hair dryer is among one of the best...,[],B0895LW9LL,B0895LW9LL,AFSKPY37N3C43SOI5IEXEK5JSIYA,1601757145389,0,False,"Jinri Professional Tourmaline Hair Dryer, Nega...",99.99,4.3,Health & Personal Care,JINRI
4,5.0,Great reasonably priced shower filter!,I live in Florida where hard water is definite...,[],B07PKLN99K,B07PKLN99K,AFSKPY37N3C43SOI5IEXEK5JSIYA,1559421381258,0,False,Gophra 15 Stage Shower Filter with 1 Cartridge...,NaN,4.5,Health & Personal Care,Gophra
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,5.0,++++++,Great product. Fast service‼️,[],B086DW5NV9,B086DW5NV9,AETVE4KIR7RIAOO7GMYQJAWF5ULQ,1610493907148,0,True,NaN,NaN,NaN,NaN,NaN
19996,5.0,Kids really loved these cups,Kids really loved these cups. The large straws...,[],B074LZ86QN,B074LZ86QN,AEX5GTAP27CPOGUPHEROKFNVBRFA,1514421160945,0,True,NaN,NaN,NaN,NaN,NaN
19997,2.0,Not a pretty sight. The product is fine if you...,"I should have gotten the longer sleeve, this o...",[],B00XWU7KTY,B00XWU7KTY,AEA7UD3WMSJYNPJH5KS4NWDHI7TQ,1440450461000,3,True,NaN,NaN,NaN,NaN,NaN
19998,5.0,Five Stars,Perfect for back support.,[],B01INOKOIM,B01INOKOIM,AEQL4HDI4HISBKEROGM43EUTFASQ,1491246245000,0,True,NaN,NaN,NaN,NaN,NaN
